In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse.linalg import svds

def build_hybrid_engine(df):
    # 1. PRÉPARATION DES DONNÉES (Tags)
    # Création de la matrice de tags (One-Hot Encoding)
    tags_split = df['tags'].str.split(', ', expand=True).stack()
    tags_dummies = pd.get_dummies(tags_split).groupby(level=0).sum()
    
    # 2. PRÉPARATION DU COMPORTEMENT (Playtime)
    matrix = df.pivot_table(index='user_id', columns='title', values='playtime').fillna(0)
    
    # 3. FILTRAGE COLLABORATIF (SVD)
    matrix_values = matrix.values.astype(float)
    U, sigma, Vt = svds(matrix_values, k=50)
    sigma = np.diag(sigma)
    collaborative_preds = np.dot(np.dot(U, sigma), Vt)
    preds_df = pd.DataFrame(collaborative_preds, columns=matrix.columns, index=matrix.index)
    
    return matrix, preds_df, tags_dummies

def get_hybrid_recommendations(user_id, matrix, preds_df, tags_dummies, df_base, alpha=0.5, beta=0.2):
    # 1. Aligner les jeux : on ne garde que l'intersection des jeux présents partout
    common_games = matrix.columns.intersection(tags_dummies.index)
    
    # Sécuriser les matrices
    tags_aligned = tags_dummies.loc[common_games]
    matrix_aligned = matrix[common_games]
    preds_aligned = preds_df[common_games]
    
    # 3. Calculs
    collab_scores = preds_aligned.loc[user_id]
    user_playtime = matrix_aligned.loc[user_id]
    user_profile = user_playtime.values.dot(tags_aligned.values)
    
    if np.all(user_profile == 0):
        final_scores = collab_scores
    else:
        content_scores = cosine_similarity(user_profile.reshape(1, -1), tags_aligned.values).flatten()
        content_scores = pd.Series(content_scores, index=common_games)
        final_scores = (alpha * collab_scores) + ((1 - alpha) * content_scores)
    
    # 4. MULTIPLICATEUR DE QUALITÉ
    final_scores = final_scores * (1 + (beta * success_ratios))
    
    # 5. FILTRAGE SÉCURISÉ
    # On ne drop que les jeux qui sont VRAIMENT dans final_scores
    played = matrix.columns[matrix.loc[user_id] > 0]
    to_drop = [g for g in played if g in final_scores.index]
    final_scores = final_scores.drop(to_drop)
    
    return final_scores.sort_values(ascending=False).head(5)
    


# LOAD DATA 
steam200k = pd.read_csv("../data/steam200k.csv")

steam200k.rename(
    columns ={
        steam200k.columns[0] : "user_id",
        steam200k.columns[1] : "title",
        steam200k.columns[2] : "status",
        steam200k.columns[3] : "playtime",
        steam200k.columns[4] : "osef"
    },
    inplace=True
)

steam200k = steam200k[steam200k["status"] == "play"]

steam200k.drop(
    columns=[
        "status",
        "osef"
    ],
    inplace=True
)

games = pd.read_csv("../data/games.csv").sort_values(by="app_id", ascending=True)

games_metadata = pd.read_json("../data/games_metadata.json", lines=True)
games = pd.merge(games, games_metadata, on="app_id")

games_and_user_playtime = pd.merge(games, steam200k, on="title")
games_and_user_playtime


# --- WORKFLOW ---
# 1. Merger les données
games_and_user_playtime = pd.merge(games, steam200k, on="title")
# 2. Construire le moteur
success_ratios = games_and_user_playtime[['title', 'positive_ratio']].drop_duplicates('title').set_index('title')['positive_ratio']

matrix, preds, tags = build_hybrid_engine(games_and_user_playtime)
recos = get_hybrid_recommendations(
    games_and_user_playtime["user_id"].value_counts().index[150], 
    matrix, preds, tags, success_ratios
)

import streamlit as st
import matplotlib.pyplot as plt

def show_recommendations(recos, tags_dummies):
    st.title("🎯 Vos recommandations personnalisées")
    
    for game in recos.index:
        with st.container():
            col1, col2 = st.columns([1, 3])
            with col1:
                # Placeholder pour une image du jeu
                st.image("https://via.placeholder.com/150", caption=game)
            with col2:
                st.subheader(game)
                # Barre de score
                score = recos[game]
                st.progress(min(score, 1.0))
                st.write(f"Score de pertinence : {score:.2f}")
                
                # Chips pour les tags
                game_tags = tags_dummies.loc[game]
                active_tags = game_tags[game_tags > 0].index.tolist()
                st.write(f"Tags : {', '.join(active_tags[:5])}")

# Exemple d'appel :
show_recommendations(recos, tags)

KeyError: "None of [Index(['title', 'positive_ratio'], dtype='str', name='title')] are in the [index]"